In [ ]:
import json
from datasets import load_dataset

In [ ]:
datasets = load_dataset('hate-speech-portuguese/hate_speech_portuguese', split='train[:10%]')


In [ ]:
print(datasets)

In [ ]:
datasets = datasets.remove_columns([
    'hatespeech_G1', 'annotator_G1', 'hatespeech_G2', 'annotator_G2', 'hatespeech_G3', 'annotator_G3'
])

In [ ]:
print(datasets)

In [ ]:
datasets = datasets.train_test_split(test_size=0.2)

In [ ]:
print(datasets)

In [ ]:
datasets["train"]["text"]

In [ ]:
def removeN(example):
    example['text'] = example['text'].replace("\n", " ")
    return example

In [ ]:
datasets = datasets.map(removeN)

In [ ]:
datasets['train'][3]

In [ ]:
# label 0 -> No hate speech
# label 1 -> Hate speech

def labelChange(example):
    example["label_text"] = 'No Hate Speech' if example['label'] == 0 else 'Hate Speech'
    return example


In [ ]:
datasets = datasets.map(labelChange)

In [ ]:
datasets = datasets.remove_columns(['label'])

In [ ]:
print(datasets['train'][0])

In [ ]:
# CONSTRUCAO DO OBJETO PARA OPENAI

def dataset_to_jsonl(dataset, file_name):
    with open(file_name, 'w', encoding='utf-8') as f:
        for example in dataset:
            json_obj = {"messages": [
                {"role": "system", "content": "Seu trabalho é classificar os comentários do usuário em Hate Speech e No Hate Speech."},
                {"role": "user", "content": example['text']},
                {"role": "assistant", "content": example['label_text']},
            ]}
            f.write(json.dumps(json_obj, ensure_ascii=False) + '\n')

In [ ]:
dataset_to_jsonl(datasets['train'], 'train.jsonl')

In [ ]:
dataset_to_jsonl(datasets['test'], 'validation.jsonl')

In [ ]:
from openai import OpenAI
import os

In [ ]:
os.environ["OPENAI_API_KEY"] = ""

In [ ]:
client = OpenAI()

In [ ]:
client.files.create(
    file=open("train.jsonl", 'rb'),
    purpose="fine-tune",
)

In [ ]:
client.files.create(
    file=open("validation.jsonl", 'rb'),
    purpose="fine-tune",
)

In [ ]:
# Deprecated

client.fine_tuning.jobs.create(
    training_file='file-5Z8ugesCePd1b8WV9pDNC7',
    validation_file='file-BoA2Thq2W73rpuyvD1bWtP',
    model='gpt-3.5-turbo'
)

In [ ]:
# CONSTRUCAO DO OBJETO PARA AWS BEDROCK

def dataset_to_jsonlAWS(dataset, file_name):
    with open(file_name, 'w', encoding='utf-8') as f:
        for example in dataset:
            json_obj = {
                "prompt": example['text'],
                "completion": example['label_text']
            }
            f.write(json.dumps(json_obj, ensure_ascii=False) + '\n')

In [ ]:
dataset_to_jsonlAWS(datasets['train'], 'train.jsonl')

In [ ]:

dataset_to_jsonlAWS(datasets['test'], 'validation.jsonl')